# 05 · Mortality forecasting: Lee-Carter to deep learning
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/actuarialsociety/deep-actuarial-science/blob/main/notebooks/05_mortality.ipynb)

$\log m_{x,t}=a_x+b_xk_t$ versus CNN, LSTM and Transformer forecasters on 10-year windows of Swiss mortality (HMD). About 5 minutes on a CPU.

_Runs on a Colab **GPU runtime** (T4), selected automatically when you open it; if Colab gives you a CPU runtime, the first cell stops and says how to switch._

In [ ]:
# Setup: works in Colab (GPU runtime), the dev container and a local clone.
import os, sys
# Colab must run on a GPU runtime: stop here otherwise (set DAS_ALLOW_CPU=1 to override).
if "google.colab" in sys.modules and os.environ.get("DAS_ALLOW_CPU") != "1":
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "This notebook needs a GPU runtime. In Colab: Runtime > Change runtime type > "
            "T4 GPU > Save, then Runtime > Run all. (To run on CPU anyway: "
            "import os; os.environ['DAS_ALLOW_CPU'] = '1')"
        )
    print("GPU:", torch.cuda.get_device_name(0))

if "google.colab" in sys.modules and not os.path.exists("/content/deep-actuarial-science"):
    !git clone -q https://github.com/actuarialsociety/deep-actuarial-science /content/deep-actuarial-science
    !pip install -q -e /content/deep-actuarial-science
if "google.colab" in sys.modules:
    os.chdir("/content/deep-actuarial-science")
    sys.path.insert(0, "src")
os.environ.setdefault("DO_NOT_TRACK", "1")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

from deep_actuarial_science.notebook import run, load, show, leaderboard, parameters, mlflow_ui
FAST = os.environ.get("DAS_ENV") == "smoke"   # CI runs every notebook in smoke mode


In [ ]:
models = ["lee_carter", "cnn", "cnn_tanh", "lstm", "transformer"]
run("mortality")
leaderboard("mortality", models)

## Open the box: Lee-Carter parameters

In [ ]:
import matplotlib.pyplot as plt
h = load("lee_carter.mort_history")["Male"]
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
ax[0].plot(h["ax"]); ax[0].set_title("a_x"); ax[1].plot(h["bx"]); ax[1].set_title("b_x")
ax[2].plot(h["kt"]); ax[2].set_title("k_t (incl. forecast)"); plt.show()

In [ ]:
show("mortality", None, width=620)

In [ ]:
show("mortality", "lstm", ["residuals", "training"])

## Production checks
* The reference notebook scales log-mortality using **all** years, including the test period: a look-ahead. Set `mortality.training.scaler_fit: learn` and re-run.
* Forecasts are recursive: errors compound with the horizon (see `rmse_by_horizon`).

## Exercise
Run both sexes (`run("mortality", env="scale_up")`) and move the forecast origin: `params={"mortality.data.obs_year": 2009}`. Which model degrades least?

## Explore the runs in MLflow
Parameters, metrics, figures and the benchmark table of every run above. In Colab the UI opens below and in a new tab through Colab's proxy (MLflow runs on the Colab machine, so a plain localhost link would point at your own computer).

In [ ]:
mlflow_ui()